# T-05 — Retrieval-Augmented Generation and the Study of Hallucination
## Phase 1 — Low-Code Baseline (YadYar Lite)

**Project:** YadYar Lite — Lightweight ML Learning Assistant  
**Topic:** T-05 — Retrieval-Augmented Generation and the Study of Hallucination  
**Phase:** Phase 1 only (Topic, Dataset, Baseline, Evaluation Plan, Analysis Plan)

---

## 0. Project Overview

This notebook builds a **lightweight Retrieval-Augmented Generation (RAG)**
pipeline on a small subset of **SQuAD 2.0** and prepares the ground for
studying **hallucination** behaviour in Phase 2.

The pipeline has three small parts:

1. **Retriever** — `sentence-transformers/all-MiniLM-L6-v2` embeds contexts; FAISS
   returns the top-k most similar contexts for a question.
2. **Generator** — `google/flan-t5-small` reads the question (optionally with a
   retrieved context) and writes a short answer.
3. **Two modes** — `generate_with_rag` (retriever + generator) and
   `generate_without_rag` (generator alone, the **No-RAG baseline**).

### What this notebook is

A small, reproducible, low-code baseline for T-05. The goal is to set up a
minimal RAG pipeline whose hallucination behaviour we can study in Phase 2 —
nothing more. We use only pretrained, off-the-shelf models; we do not train
or fine-tune anything.

### What this notebook is NOT

- It is **not** a product, an API, or a UI.
- It is **not** Phase 2 — full evaluation and error analysis are out of scope here.
- It does **not** train any model from scratch.
- It does **not** perform fine-tuning.
- It does **not** run full evaluation or final error analysis — that is **Phase 2**.

Phase 1 only requires: a narrowed question, a documented dataset, a reproducible
baseline, a short evaluation plan, and a simple analysis plan. This notebook
delivers exactly those, plus a tiny sanity check so we know the code runs.

## 1. Narrow Project Question

> **Can a lightweight RAG pipeline using dense retrieval reduce unsupported or
> hallucinated answers compared with a no-retrieval baseline on a small subset
> of SQuAD 2.0?**

**توضیح ساده (فارسی):**

آیا یک پایپ‌لاین RAG سبک که از dense retrieval (جستجوی معنایی با embedding) استفاده
می‌کند، می‌تواند تعداد پاسخ‌های بی‌پشتوانه یا hallucinated (ساخته‌شده توسط مدل) را
نسبت به حالت بدون بازیابی (No-RAG) روی یک زیرمجموعه کوچک از SQuAD 2.0 کاهش دهد؟

این سؤال محدود، قابل اجرا و قابل دفاع است زیرا:

- فقط یک مدل سبک آماده استفاده می‌شود (no training, no fine-tuning).
- فقط یک زیرمجموعه کوچک از یک دیتاست عمومی بررسی می‌شود.
- فقط دو حالت مقایسه می‌شود: **RAG** در برابر **No-RAG**.
- خروجی قابل اندازه‌گیری است: در فاز دوم با معیارهایی مثل Exact Match،
  Token F1 و Abstention Rate پاسخ‌ها ارزیابی می‌شوند.

این سؤال مستقیماً به الگوهای شکست ذکرشده در صورت پروژه برای T-05 اشاره می‌کند:
**absent-evidence hallucination** و **ignored-evidence hallucination**.

## 2. Dataset Description

| Field | Value |
|---|---|
| Dataset | **SQuAD 2.0** (Stanford Question Answering Dataset v2) |
| Source | Hugging Face Datasets: `rajpurkar/squad_v2` |
| Split used | `train` (a small subset of it) |
| Subset size | ~300 unique contexts, ~80 questions (mixed answerable / unanswerable) |

### Important fields

| Field | Meaning |
|---|---|
| `context` | The passage (paragraph) from which the answer should be extracted |
| `question` | The student-style question |
| `answers["text"]` | A list of gold answer strings (may be **empty** → unanswerable) |
| `answers["answer_start"]` | Character offsets of each gold answer in the context |

### ⚠️ Note on unanswerable questions

In the Hugging Face version of `squad_v2`, the `is_impossible` column is **not**
guaranteed to exist. The robust way to detect an unanswerable question is:

```python
is_unanswerable = len(example["answers"]["text"]) == 0
```

### Why SQuAD 2.0 fits T-05

- It is **public, small, well-documented, and widely used** for QA / RAG studies.
- It contains **both answerable and unanswerable** questions, which lets us study
  hallucination under two failure regimes: (a) the answer is in the context but the
  model misses it, and (b) the answer is **not** in the context, which is exactly
  where absent-evidence hallucination can appear.
- Each context is short (a paragraph), so embeddings and FLAN-T5-small both fit
  comfortably in a free Colab runtime.
- It is one of the suggested starting points for T-05 in the project catalogue.

## 3. Baseline Setup

### 3.1 Components

| Role | Tool / Model | Notes |
|---|---|---|
| Dataset loader | `datasets` (Hugging Face) | Loads `rajpurkar/squad_v2` |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` | 384-dim, fast, CPU-friendly |
| Vector search | `faiss-cpu` (flat L2 index) | Simplest explainable ANN baseline |
| Generator (RAG) | `google/flan-t5-small` | ~80M params, runs on free Colab |
| No-RAG baseline | Same `flan-t5-small` **without** retrieved context | Apples-to-apples comparison |

### 3.2 Reproducibility

- Random seed is fixed (`SEED = 42`).
- Subset selection is deterministic given the seed.
- Models are pinned by their Hugging Face IDs.
- The whole pipeline runs in a free Colab CPU runtime in a few minutes.

### 3.3 Install dependencies

In [1]:
# Run once per Colab session. ~1–2 minutes on a fresh runtime.
!pip install -q datasets sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 75.7 MB/s eta 0:00:00


### 3.4 Imports and seed

We import the four building blocks: dataset loading, embeddings, FAISS, and the
FLAN-T5 generator. A fixed seed makes the subset selection reproducible.

In [2]:
import os, random, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import T5ForConditionalGeneration, T5TokenizerFast

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Seed fixed:", SEED)

Seed fixed: 42


### 3.5 Load SQuAD 2.0

We load only the `train` split. The full split has ~130k examples, but we will
keep only a small subset in the next cell.

In [3]:
squad = load_dataset("rajpurkar/squad_v2", split="train")
print("Total SQuAD v2 train rows:", len(squad))
print("Sample row keys:", list(squad[0].keys()))
print("First context (truncated):", squad[0]["context"][:200])
print("First question:", squad[0]["question"])
print("First answers:", squad[0]["answers"])

README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Total SQuAD v2 train rows: 130319
Sample row keys: ['id', 'title', 'context', 'question', 'answers']
First context (truncated): Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in v
First question: When did Beyonce start becoming popular?
First answers: {'text': ['in the late 1990s'], 'answer_start': [269]}


### 3.6 Build a small, balanced subset

Goal: ~300 unique contexts and ~80 questions, with a mix of answerable and
unanswerable examples. We:

1. Shuffle the dataset deterministically.
2. Walk through it and collect **unique contexts** until we have ~300.
3. From the rows whose context is in that set, pick ~80 questions, making sure
   at least ~25% are unanswerable (empty `answers["text"]`) so Phase 2 can study
   absent-evidence hallucination.

In [4]:
TARGET_CONTEXTS = 300
TARGET_QUESTIONS = 80
TARGET_UNANSWERABLE = 20   # at least ~25% of the questions

# Deterministic shuffle
idxs = list(range(len(squad)))
random.Random(SEED).shuffle(idxs)

# Step 1: collect ~300 unique contexts
seen_contexts = {}          # context -> first row index
for i in idxs:
    ctx = squad[i]["context"]
    if ctx not in seen_contexts:
        seen_contexts[ctx] = i
    if len(seen_contexts) >= TARGET_CONTEXTS:
        break
contexts_list = list(seen_contexts.keys())
print("Unique contexts selected:", len(contexts_list))

# Step 2: gather candidate questions whose context is in our set
ctx_set = set(contexts_list)
cand_answerable, cand_unanswerable = [], []
for i in idxs:
    row = squad[i]
    if row["context"] not in ctx_set:
        continue
    is_unans = len(row["answers"]["text"]) == 0
    if is_unans:
        cand_unanswerable.append(i)
    else:
        cand_answerable.append(i)

# Step 3: pick a balanced subset
n_unans = min(TARGET_UNANSWERABLE, len(cand_unanswerable))
n_ans = TARGET_QUESTIONS - n_unans
selected = (cand_answerable[:n_ans] + cand_unanswerable[:n_unans])
random.Random(SEED).shuffle(selected)

subset = squad.select(selected)
print(f"Subset questions: {len(subset)} "
      f"(answerable={n_ans}, unanswerable={n_unans})")

Unique contexts selected: 300
Subset questions: 80 (answerable=60, unanswerable=20)


### 3.7 Embed contexts

We load `all-MiniLM-L6-v2` and embed the ~300 contexts. The model returns 384-dim
L2-normalised vectors, which work directly with FAISS.

In [5]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding dim:", embedder.get_sentence_embedding_dimension())

context_embeddings = embedder.encode(
    contexts_list,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Context embeddings shape:", context_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384


/tmp/ipykernel_1918/1726465436.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dim:", embedder.get_sentence_embedding_dimension())


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Context embeddings shape: (300, 384)


### 3.8 Build FAISS index

We use a flat L2 index — the simplest possible vector search. There is no
approximation (no IVF, no PQ), so retrieval is exact and easy to explain in
the oral defence.

In [6]:
dim = context_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(context_embeddings)
print("FAISS index size (vectors):", index.ntotal)

FAISS index size (vectors): 300


### 3.9 Retrieve function

Given a question, embed it, search the FAISS index, and return the top-k
context strings together with their indices.

In [7]:
def retrieve(question: str, k: int = 5):
    """Return top-k contexts for a question."""
    q_emb = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        results.append({
            "rank": rank,
            "score": float(dist),
            "context_idx": int(idx),
            "context": contexts_list[idx],
        })
    return results

# Quick test on one question
sample_q = subset[0]["question"]
print("Question:", sample_q)
for r in retrieve(sample_q, k=3):
    print(f"  rank={r['rank']}  L2={r['score']:.4f}  ctx={r['context'][:80]}...")

Question: What was the name for a pub that could sell beer from more than one brewery?
  rank=1  L2=0.7893  ctx=After the development of the large London Porter breweries in the 18th century, ...
  rank=2  L2=1.4305  ctx=There are seven current masjids in the Greater Richmond area, with three more cu...
  rank=3  L2=1.4352  ctx="Whereas their Majesties have been Graciously Pleased to grant Letters patent to...


### 3.10 Load FLAN-T5-small generator

We use `google/flan-t5-small` (~80M params). It is small enough to run on a free
Colab CPU in seconds per question, and is instruction-tuned, so it follows short
prompts like "answer the question using the context".

In [8]:
MODEL_ID = "google/flan-t5-small"
tokenizer = T5TokenizerFast.from_pretrained(MODEL_ID)
generator = T5ForConditionalGeneration.from_pretrained(MODEL_ID)
print("Loaded:", MODEL_ID)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded: google/flan-t5-small


### 3.11 Generation functions (RAG and No-RAG)

Both modes use the **same** generator so any difference in answer quality comes
from the retrieved context, not from a different model.

- `generate_with_rag(question)` — retrieves top-1 context and asks FLAN-T5 to
  answer using it. If the model thinks the context does not contain the answer,
  it is free to say "I don't know" (we do not force an answer).
- `generate_without_rag(question)` — passes only the question to FLAN-T5. This
  is the baseline that is most likely to hallucinate when the answer needs a
  specific fact from the corpus.

In [9]:
def _generate(prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    out = generator.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,            # greedy for reproducibility
        num_beams=1,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def generate_with_rag(question: str, k: int = 1):
    retrieved = retrieve(question, k=k)
    top_ctx = retrieved[0]["context"]
    prompt = (
        "Answer the question using the context. "
        "If the context does not contain the answer, say: I don't know.\n\n"
        f"Context: {top_ctx}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    answer = _generate(prompt)
    return {"answer": answer, "retrieved_context": top_ctx,
            "all_retrieved": retrieved}

def generate_without_rag(question: str):
    prompt = f"Answer the question.\n\nQuestion: {question}\n\nAnswer:"
    answer = _generate(prompt)
    return {"answer": answer, "retrieved_context": None}

### 3.12 Sanity check (Phase 1 only)

We run **three** sample questions to confirm the pipeline works end-to-end:

1. One **answerable** question — compare RAG vs No-RAG.
2. One **unanswerable** question — see whether the RAG mode abstains or hallucinates.
3. One more **answerable** question for variety.

This is **not** evaluation — it is just a smoke test. Real metrics live in Phase 2.

In [10]:
def find_one(answerable: bool):
    for row in subset:
        is_unans = len(row["answers"]["text"]) == 0
        if answerable and not is_unans:
            return row
        if (not answerable) and is_unans:
            return row
    return None

samples = [find_one(answerable=True), find_one(answerable=False),
           find_one(answerable=True)]

for i, row in enumerate(samples, start=1):
    q = row["question"]
    gold = row["answers"]["text"]
    gold_str = gold[0] if gold else "(unanswerable — gold is empty)"
    rag = generate_with_rag(q)
    norag = generate_without_rag(q)
    print(f"\n===== Sample {i} =====")
    print("Question      :", q)
    print("Gold          :", gold_str)
    print("RAG context   :", rag["retrieved_context"][:140].replace('\n', ' '), "...")
    print("RAG answer    :", rag["answer"])
    print("No-RAG answer :", norag["answer"])


===== Sample 1 =====
Question      : What was the name for a pub that could sell beer from more than one brewery?
Gold          : a Free house
RAG context   : After the development of the large London Porter breweries in the 18th century, the trend grew for pubs to become tied houses which could on ...
RAG answer    : a Free house
No-RAG answer : st johns

===== Sample 2 =====
Question      : Nomadic hunter-gatherers are an exception to what rule?
Gold          : (unanswerable — gold is empty)
RAG context   : Hunter-gatherers tend to have an egalitarian social ethos, although settled hunter-gatherers (for example, those inhabiting the Northwest Co ...
RAG answer    : unanswerable
No-RAG answer : stagiation

===== Sample 3 =====
Question      : What was the name for a pub that could sell beer from more than one brewery?
Gold          : a Free house
RAG context   : After the development of the large London Porter breweries in the 18th century, the trend grew for pubs to become tied hous

## 4. Evaluation Plan (for Phase 2)

Phase 1 only requires a **plan**. We will compute the following metrics in Phase 2
on the same subset. Below each metric is a one-line explanation of what it measures.

### 4.1 Retrieval metrics

| Metric | What it measures |
|---|---|
| **Recall@5** | Of the gold context, how often is it inside the top-5 retrieved contexts? |
| **MRR** (Mean Reciprocal Rank) | How high is the gold context in the retrieved list, on average (1/rank)? |

### 4.2 Generation metrics

| Metric | What it measures |
|---|---|
| **Exact Match (EM)** | Fraction of answers that exactly match the gold (after normalisation) |
| **Token F1** | Token-level F1 between predicted and gold answer — partial credit |
| **ROUGE-L** | Longest common subsequence overlap between prediction and gold |

### 4.3 Hallucination-specific metric

| Metric | What it measures |
|---|---|
| **Abstention rate** | On unanswerable questions, the fraction where the model says "I don't know" instead of inventing an answer |
| **Hallucination rate** (manual, small sample) | On a sample of ~30 RAG answers, the fraction judged by a human as unsupported by the retrieved context |

### 4.4 Sanity check (already run above)

The 3-question smoke test in §3.12 confirms:

- the dataset loads and the subset is built;
- embeddings and the FAISS index work;
- both `generate_with_rag` and `generate_without_rag` produce a string;
- the No-RAG baseline sometimes invents facts when RAG abstains (or vice versa).

That is enough for Phase 1.

## 5. Simple Analysis Plan (for Phase 2)

We will group errors into **four categories** (the first three come directly from
the T-05 catalogue entry; the fourth is a data slice that is easy to compute on
SQuAD).

### 5.1 Error categories

| # | Error type | Definition |
|---|---|---|
| 1 | **Retrieval miss / absent-evidence hallucination** | The gold context was not in top-k, so the generator either hallucinates or abstains. |
| 2 | **Ignored-evidence hallucination** | The gold context *was* retrieved, but the generator still produced an answer unsupported by it. |
| 3 | **Unanswerable failure** | The question is unanswerable, but the model produced a confident (hallucinated) answer instead of abstaining. |
| 4 | **Short question vs long question** | Performance breakdown by question length (≤ 8 tokens vs > 8 tokens) — short questions may retrieve more weakly. |

### 5.2 Error-analysis template (Phase 2)

We will fill in the following small table for ~20 representative failures:

| Question | Gold answer | Retrieved context correct? | RAG answer | No-RAG answer | Error type | Explanation |
|---|---|---|---|---|---|---|
| _(filled in Phase 2)_ | | | | | | |

Each row will be short (one sentence per cell). The point is qualitative
understanding, not formal statistics.

## 6. Scope and Limitations

- This is **Phase 1 only**. We deliver a baseline and an evaluation plan, not
  final numbers.
- The subset is **very small** (~300 contexts, ~80 questions). Numbers from
  Phase 2 will be indicative, not statistically rigorous.
- The generator is **`flan-t5-small`** (~80M params). It is intentionally weak so
  that hallucination cases are visible and easy to discuss.
- The prompt is **a single, simple template**. No prompt engineering, no
  chain-of-thought, no self-consistency.
- Retrieval is **flat L2 with no reranker**. There is no approximate NN search
  and no hybrid (keyword + dense) retrieval.
- **Hallucination detection in Phase 1 is rule-based and preliminary** — we
  detect abstention with a substring rule (`"i don't know"` in the lower-cased
  answer). A proper human-judged hallucination rate is part of Phase 2.
- **Unanswerable detection** relies on `len(answers["text"]) == 0` because the
  Hugging Face version of `squad_v2` does not reliably expose `is_impossible`.

# Phase 2 — Evaluation, Error Analysis, and Lightweight Demo

This section is a **direct continuation of Phase 1** in the same notebook.
It reuses the same SQuAD 2.0 corpus, MiniLM embeddings, FAISS index,
`google/flan-t5-small`, RAG prompt, and No-RAG prompt.

Phase 2 only evaluates the existing baseline, analyses errors, runs a small
demo, and saves the real outputs. It does not introduce a new model or a
second implementation of Phase 1.


## Phase 2.1 — Verify Phase 1 Objects

This cell checks that the unchanged Phase 1 dataset, retriever, index, generator, and helper functions are available. Run the notebook from the beginning before Phase 2.

In [11]:
# Phase 2 must reuse the objects created by the unchanged Phase 1 cells.
required_phase1_objects = [
    "squad", "subset", "contexts_list", "embedder",
    "context_embeddings", "index", "tokenizer", "generator",
    "retrieve", "generate_with_rag", "generate_without_rag",
]
missing = [name for name in required_phase1_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the notebook from the beginning. Missing Phase 1 objects: "
        + ", ".join(missing)
    )

print("Phase 1 objects found; Phase 2 will reuse the original baseline.")
print("Model:", MODEL_ID, "| corpus:", len(contexts_list), "contexts")


Phase 1 objects found; Phase 2 will reuse the original baseline.
Model: google/flan-t5-small | corpus: 300 contexts


## Phase 2.2 — Evaluation Configuration

`N_EVAL=30` provides a fast run; increase it to 100 for a fuller evaluation.

The evaluation is **closed-corpus**: only questions whose gold context already exists in the 300-context Phase 1 index are sampled. Retrieval scores therefore measure ranking within this small corpus, not performance on all of SQuAD 2.0.

In [12]:
# ---- Phase 2 evaluation configuration ----
N_EVAL = 30                 # set to 100 for a fuller, slower run
TOP_K_VALUES = [1, 3, 5]    # retriever diagnostics
RANDOM_SEED = 42
RETRIEVE_K = 1              # Phase 1 feeds only top-1 to the generator
CORRECT_F1_THRESHOLD = 0.80
CORPUS_SIZE = len(contexts_list)

import os, json, re, string, random, shutil
from collections import Counter
import numpy as np
import pandas as pd

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

OUTPUT_DIR = "./t05_phase2_outputs"
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Corpus size:", CORPUS_SIZE)
print("N_EVAL:", N_EVAL)
print("RAG contexts actually used:", RETRIEVE_K)
print("Retrieval diagnostics:", TOP_K_VALUES)
print("OUTPUT_DIR:", OUTPUT_DIR)


Corpus size: 300
N_EVAL: 30
RAG contexts actually used: 1
Retrieval diagnostics: [1, 3, 5]
OUTPUT_DIR: ./t05_phase2_outputs


### Build a closed-corpus evaluation subset

Select evaluation rows by their unique SQuAD IDs, preserve all gold answers, and keep an approximately 75/25 answerable–unanswerable split.

In [13]:
ctx_set = set(contexts_list)
phase1_ids = set(subset["id"])

candidates = [
    row for row in squad
    if row["context"] in ctx_set and row["id"] not in phase1_ids
]
rng = random.Random(RANDOM_SEED)
rng.shuffle(candidates)

answerable = [r for r in candidates if len(r["answers"]["text"]) > 0]
unanswerable = [r for r in candidates if len(r["answers"]["text"]) == 0]

n_unans = min(N_EVAL // 4, len(unanswerable))
n_ans = min(N_EVAL - n_unans, len(answerable))
eval_rows = answerable[:n_ans] + unanswerable[:n_unans]

# Fill any shortfall from unused candidates while preserving real data only.
if len(eval_rows) < N_EVAL:
    used_ids = {r["id"] for r in eval_rows}
    extras = [r for r in candidates if r["id"] not in used_ids]
    eval_rows.extend(extras[:N_EVAL - len(eval_rows)])

rng.shuffle(eval_rows)
assert eval_rows, "No closed-corpus evaluation rows were found."

n_answerable = sum(len(r["answers"]["text"]) > 0 for r in eval_rows)
print("Candidates:", len(candidates))
print(f"Evaluation rows: {len(eval_rows)} "
      f"(answerable={n_answerable}, unanswerable={len(eval_rows)-n_answerable})")


Candidates: 2329
Evaluation rows: 30 (answerable=23, unanswerable=7)


## Phase 2.3 — Retrieval Evaluation

For each answerable question, retrieve the top 5 contexts once and compute
Recall@1/3/5 and MRR by exact equality with the gold SQuAD context.

**Interpretation:** because every gold context is deliberately inside the
Phase 1 index, these numbers evaluate ranking quality *within this corpus*.
Recall@1 is the only retrieval metric directly aligned with generation,
because the unchanged Phase 1 RAG function supplies only top-1 evidence.


In [14]:
def evaluate_retrieval(rows, top_k_values):
    """Single retrieve() call per row. Returns per-k Recall@k, MRR and
    a per-question list with the retrieved contexts and the gold rank."""
    k_max = max(top_k_values)
    hit_counts = {k: 0 for k in top_k_values}
    ranks = []
    per_question = []
    n = len(rows)
    for row in rows:
        gold_ctx = row["context"]
        retrieved = retrieve(row["question"], k=k_max)
        retrieved_ctxs = [r["context"] for r in retrieved]
        rank = 0
        for i, ctx in enumerate(retrieved_ctxs, start=1):
            if ctx == gold_ctx:
                rank = i
                break
        ranks.append(rank)
        for k in top_k_values:
            if 0 < rank <= k:
                hit_counts[k] += 1
        per_question.append({
            "question": row["question"],
            "gold_context": gold_ctx,
            "retrieved_top_k": retrieved_ctxs,
            "retrieved_top_k_scores": [r["score"] for r in retrieved],
            "gold_rank": rank,
            "retrieval_success_top1": (rank == 1),
        })
    recall = {f"Recall@{k}": (hit_counts[k] / n if n else 0.0) for k in top_k_values}
    mrr = float(np.mean([1.0 / r if r > 0 else 0.0 for r in ranks])) if n else 0.0
    summary = {**recall, "MRR": mrr, "n_questions": n}
    return summary, per_question

answerable_eval = [r for r in eval_rows if len(r["answers"]["text"]) > 0]
print("Answerable questions used for retrieval evaluation:", len(answerable_eval))

retrieval_summary, retrieval_per_q = evaluate_retrieval(answerable_eval, TOP_K_VALUES)
retrieval_metrics = dict(retrieval_summary)
print(json.dumps(retrieval_summary, indent=2))

retrieval_summary_df = pd.DataFrame([{
    "Recall@1":  retrieval_summary["Recall@1"],
    "Recall@3":  retrieval_summary["Recall@3"],
    "Recall@5":  retrieval_summary["Recall@5"],
    "MRR":       retrieval_summary["MRR"],
    "n_questions": retrieval_summary["n_questions"],
}])
retrieval_summary_df


Answerable questions used for retrieval evaluation: 23
{
  "Recall@1": 0.9130434782608695,
  "Recall@3": 1.0,
  "Recall@5": 1.0,
  "MRR": 0.9565217391304348,
  "n_questions": 23
}


,Recall@1,Recall@3,Recall@5,MRR,n_questions
0,0.913043,1.0,1.0,0.956522,23


### Retrieval metrics — interpretation

- **Recall@1** measures how often the gold context is ranked first. It is the retrieval metric aligned with generation because Phase 1 passes only the top-1 context to the generator.
- **Recall@3 / Recall@5** are retriever diagnostics; they do not mean the generator saw those passages.
- **MRR** rewards placing the gold context near the top, from 0 to 1.

These values apply only to the closed 300-context corpus.

## Phase 2.4 — Generation Evaluation

RAG and No-RAG reuse the unchanged Phase 1 functions and the same `flan-t5-small` generator. Generation-time retrieval success is computed from the exact top-1 context actually passed to the RAG model.

In [15]:
# ---- SQuAD-style lexical scoring helpers ----
ARTICLES = {"a", "an", "the"}


def normalize_text(text: str) -> str:
    text = (text or "").lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    return " ".join(t for t in text.split() if t not in ARTICLES)


def _em_one(pred: str, gold: str) -> int:
    return int(normalize_text(pred) == normalize_text(gold))


def _f1_one(pred: str, gold: str) -> float:
    pred_tokens, gold_tokens = normalize_text(pred).split(), normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    overlap = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if not overlap:
        return 0.0
    precision, recall = overlap / len(pred_tokens), overlap / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def _rouge_l_one(pred: str, gold: str) -> float:
    pred_tokens, gold_tokens = normalize_text(pred).split(), normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)
    previous = [0] * (len(gold_tokens) + 1)
    for p in pred_tokens:
        current = [0]
        for j, g in enumerate(gold_tokens, 1):
            current.append(previous[j-1] + 1 if p == g else max(previous[j], current[-1]))
        previous = current
    lcs = previous[-1]
    if not lcs:
        return 0.0
    precision, recall = lcs / len(pred_tokens), lcs / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def score_against_golds(pred: str, golds):
    golds = list(golds) if golds else [""]
    return (
        max(_em_one(pred, gold) for gold in golds),
        max(_f1_one(pred, gold) for gold in golds),
        max(_rouge_l_one(pred, gold) for gold in golds),
    )


NO_ANSWER_PHRASES = (
    "i don't know", "i do not know", "no answer", "unknown", "unanswerable",
    "not enough information", "insufficient information", "cannot be answered",
    "can't be answered", "cannot determine", "not in the context",
    "context does not contain",
)


def is_abstention(answer: str) -> bool:
    normalized = normalize_text(answer)
    return not normalized or any(phrase.replace("'", "") in normalized
                                 for phrase in NO_ANSWER_PHRASES)

print("Scoring and abstention helpers ready.")


Scoring and abstention helpers ready.


In [16]:
# ---- Run the unchanged Phase 1 RAG and No-RAG baselines ----
import torch

_retrieval_by_question = {r["question"]: r for r in retrieval_per_q}
records = []

for i, row in enumerate(eval_rows, start=1):
    question = row["question"]
    gold_texts = list(row["answers"]["text"])
    is_answerable = len(gold_texts) > 0

    # Top-5 is used only for retriever diagnostics.
    diagnostic = _retrieval_by_question.get(question)
    if diagnostic is None:
        retrieved = retrieve(question, k=max(TOP_K_VALUES))
        diagnostic_contexts = [r["context"] for r in retrieved]
        gold_rank = next(
            (rank for rank, ctx in enumerate(diagnostic_contexts, 1)
             if ctx == row["context"]),
            0,
        )
    else:
        diagnostic_contexts = diagnostic["retrieved_top_k"]
        gold_rank = diagnostic["gold_rank"]

    with torch.no_grad():
        rag_out = generate_with_rag(question, k=RETRIEVE_K)
        norag_out = generate_without_rag(question)

    rag_top_context = rag_out.get("retrieved_context", "")
    records.append({
        "idx": i,
        "id": row["id"],
        "question": question,
        "question_len_words": len(question.split()),
        "gold_texts": gold_texts,
        "is_answerable": is_answerable,
        "gold_context": row["context"],
        "gold_rank": gold_rank,
        "retrieval_success_top1": rag_top_context == row["context"],
        "retrieval_success_at_5": 0 < gold_rank <= 5,
        "retrieved_top5": diagnostic_contexts[:5],
        "rag_answer": rag_out["answer"],
        "norag_answer": norag_out["answer"],
        "rag_top_context": rag_top_context,
    })

    if i % 5 == 0 or i == len(eval_rows):
        print(f"...processed {i}/{len(eval_rows)}")

results_df = pd.DataFrame(records)
print("Evaluation records:", len(results_df))
results_df.head()


...processed 5/30
...processed 10/30
...processed 15/30
...processed 20/30
...processed 25/30
...processed 30/30
Evaluation records: 30


,idx,id,question,question_len_words,gold_texts,is_answerable,gold_context,gold_rank,retrieval_success_top1,retrieval_success_at_5,retrieved_top5,rag_answer,norag_answer,rag_top_context
0,1,572693cbdd62a815002e8a0c,How long did it take for Myanmar to recover fr...,17,[250 years],True,Pagan's collapse was followed by 250 years of ...,1,True,True,[Pagan's collapse was followed by 250 years of...,250 years,a few days,Pagan's collapse was followed by 250 years of ...
1,2,572ecf2503f9891900756a41,Which recitation is the original basis of the ...,11,[ʻAsim's],True,Vocalization markers indicating specific vowel...,1,True,True,[Vocalization markers indicating specific vowe...,Asim's recitation,sacrimony,Vocalization markers indicating specific vowel...
2,3,56bfbda3a10cfb140055129a,How did Etta James influence her?,6,[boldness],True,The feminism and female empowerment themes on ...,1,True,True,[The feminism and female empowerment themes on...,"""boldness""","a sexy, sexy, and sexy woman",The feminism and female empowerment themes on ...
3,4,570ac2e34103511400d599a1,What is Liaoning classifed as?,5,[a training ship],True,One STOBAR carrier: Liaoning was originally bu...,1,True,True,[One STOBAR carrier: Liaoning was originally b...,training ship,a syringe,One STOBAR carrier: Liaoning was originally bu...
4,5,57260d97ec44d21400f3d857,Where does Deloitte Football Money League rank...,13,[seventh],True,Arsenal's financial results for the 2014–15 se...,1,True,True,[Arsenal's financial results for the 2014–15 s...,seventh,adolescent,Arsenal's financial results for the 2014–15 se...


In [17]:
# ---- Per-question metrics and aggregate comparison ----

def score_row(row, answer_col):
    prediction = row[answer_col]
    abstained = is_abstention(prediction)
    if row["is_answerable"]:
        em, f1, rouge_l = score_against_golds(prediction, row["gold_texts"])
    else:
        em = int(abstained)
        f1 = rouge_l = float(em)
    return pd.Series({
        f"{answer_col}_em": int(em),
        f"{answer_col}_f1": float(f1),
        f"{answer_col}_rougeL": float(rouge_l),
        f"{answer_col}_abstained": int(abstained),
    })

results_df = pd.concat([
    results_df,
    results_df.apply(score_row, axis=1, answer_col="rag_answer"),
    results_df.apply(score_row, axis=1, answer_col="norag_answer"),
], axis=1)


def aggregate(df, answer_col, label):
    answerable = df[df["is_answerable"]]
    unanswerable = df[~df["is_answerable"]]
    return {
        "mode": label,
        "n": len(df),
        "EM": float(df[f"{answer_col}_em"].mean()) if len(df) else 0.0,
        "Token F1": float(df[f"{answer_col}_f1"].mean()) if len(df) else 0.0,
        "ROUGE-L": float(df[f"{answer_col}_rougeL"].mean()) if len(df) else 0.0,
        "Answerable EM": float(answerable[f"{answer_col}_em"].mean()) if len(answerable) else 0.0,
        "Unanswerable detection": float(unanswerable[f"{answer_col}_abstained"].mean()) if len(unanswerable) else 0.0,
    }

metrics_summary_df = pd.DataFrame([
    aggregate(results_df, "rag_answer", "RAG"),
    aggregate(results_df, "norag_answer", "No-RAG"),
])
display(metrics_summary_df)

# A visible diagnostic makes an all-zero No-RAG result auditable rather than
# silently treating it as either a bug or a valid result.
norag_preview = results_df[results_df["is_answerable"]][[
    "question", "gold_texts", "norag_answer",
    "norag_answer_em", "norag_answer_f1",
]].sample(n=min(8, int(results_df["is_answerable"].sum())),
          random_state=RANDOM_SEED)

if results_df["norag_answer_em"].sum() == 0 and results_df["norag_answer_f1"].sum() == 0:
    print("All No-RAG lexical scores are zero. Inspecting real predictions:")
display(norag_preview)


,mode,n,EM,Token F1,ROUGE-L,Answerable EM,Unanswerable detection
0,RAG,30,0.333333,0.441111,0.441111,0.434783,0.0
1,No-RAG,30,0.000000,0.000000,0.000000,0.000000,0.0


All No-RAG lexical scores are zero. Inspecting real predictions:


,question,gold_texts,norag_answer,norag_answer_em,norag_answer_f1
21,What were the Turks waiting for when positione...,[reinforcements],a slum,0.0,0.0
13,What does the abbreviation OCA stand for?,[Offensive Counterair],octain,0.0,0.0
0,How long did it take for Myanmar to recover fr...,[250 years],a few days,0.0,0.0
11,How many causalities did the US Air Force suff...,"[68,000]",2,0.0,0.0
23,"Which city did Baden, Württemberg-Baden, and W...",[Baden-Württemberg],berlin,0.0,0.0
18,Which crown did the King of Scotland inherit i...,[Crown of England],king s s s s s s s s s s s s s s s s s s s s s...,0.0,0.0
1,Which recitation is the original basis of the ...,[ʻAsim's],sacrimony,0.0,0.0
19,What can videoconferencing offer students?,[participating in two-way communication forums],videoconferencing,0.0,0.0


### Generation metrics — interpretation

- **EM** gives full credit only for a normalized exact match with any gold answer; on unanswerable rows it rewards abstention.
- **Token F1** gives partial lexical-overlap credit using the best score across all gold answers.
- **ROUGE-L** measures ordered token overlap.
- **Answerable EM** evaluates answerable rows only.
- **Unanswerable detection** measures how often the model abstains instead of inventing an answer.

An all-zero No-RAG result is possible with this small generator because the No-RAG prompt contains no passage. The notebook therefore displays raw No-RAG predictions so the result can be checked directly.

## Phase 2.5 — Simple Result Breakdown

The tables compare answerable/unanswerable rows, short/long questions, and
answerable questions with successful/failed top-1 retrieval. For the
unanswerable slice, EM/F1 equal the abstention success rate by design.


In [18]:
def breakdown(df, mask, label):
    sub = df[mask]
    n = len(sub)
    if n == 0:
        return {"slice": label, "n": 0,
                "RAG EM": 0.0, "RAG F1": 0.0,
                "NoRAG EM": 0.0, "NoRAG F1": 0.0}
    return {
        "slice": label, "n": n,
        "RAG EM":   float(sub["rag_answer_em"].mean()),
        "RAG F1":   float(sub["rag_answer_f1"].mean()),
        "NoRAG EM": float(sub["norag_answer_em"].mean()),
        "NoRAG F1": float(sub["norag_answer_f1"].mean()),
    }

median_len = float(results_df["question_len_words"].median())
print("Median question length (words):", median_len)

breakdown_rows = []
# 1. Answerable vs Unanswerable
breakdown_rows.append(breakdown(results_df, results_df["is_answerable"], "Answerable"))
breakdown_rows.append(breakdown(results_df, ~results_df["is_answerable"], "Unanswerable"))
# 2. Short vs Long (by median length)
breakdown_rows.append(breakdown(results_df,
                                results_df["question_len_words"] <= median_len,
                                f"Short (<= {int(median_len)} words)"))
breakdown_rows.append(breakdown(results_df,
                                results_df["question_len_words"] > median_len,
                                f"Long (> {int(median_len)} words)"))
# 3. Retrieval success vs failure (answerable only — unanswerable rows
#    have a gold context by construction, so the comparison is fair
#    only for answerable questions).
ans_df = results_df[results_df["is_answerable"]]
breakdown_rows.append(breakdown(ans_df, ans_df["retrieval_success_top1"],
                                "Answerable & retrieval success (top-1)"))
breakdown_rows.append(breakdown(ans_df, ~ans_df["retrieval_success_top1"],
                                "Answerable & retrieval failure (top-1)"))

breakdown_df = pd.DataFrame(breakdown_rows)
breakdown_df


Median question length (words): 10.0


,slice,n,RAG EM,RAG F1,NoRAG EM,NoRAG F1
0,Answerable,23,0.434783,0.575362,0.0,0.0
1,Unanswerable,7,0.000000,0.000000,0.0,0.0
2,Short (<= 10 words),20,0.350000,0.478333,0.0,0.0
3,Long (> 10 words),10,0.300000,0.366667,0.0,0.0
4,Answerable & retrieval success (top-1),21,0.476190,0.630159,0.0,0.0
5,Answerable & retrieval failure (top-1),2,0.000000,0.000000,0.0,0.0


### Breakdown — interpretation

Use the table above to compare:

- answerable versus unanswerable questions,
- short versus long questions,
- successful versus failed top-1 retrieval.

The last comparison shows whether RAG answer quality depends on retrieving the gold context. These slices are descriptive and are not formal statistical tests.

## Phase 2.6 — Hallucination and Error Analysis

We assign every RAG answer to one of five categories using a transparent
rule-based classifier. **This is not a substitute for human evaluation**;
it is a quick, defensible labelling that makes the dominant failure mode
visible for the report.

### Categories

| # | Label | Rule |
|---|-------|------|
| 1 | `Correct` | Answerable and (EM == 1 or Token F1 >= 0.80). |
| 2 | `Retrieval miss / absent-evidence hallucination` | Answerable, the gold context was **not** the top-1 context the generator saw (`retrieval_success_top1 == False`), and the model produced a non-abstention answer. The model wrote an answer without the correct passage in view. |
| 3 | `Ignored-evidence hallucination` | Answerable, the gold context **was** the top-1 context the generator saw (`retrieval_success_top1 == True`), but the answer is wrong (EM == 0 and F1 < 0.80). The model had the evidence and did not use it. |
| 4 | `Unanswerable failure` | Unanswerable question, but the model produced a non-abstention answer (it invented content). |
| 5 | `No-answer / abstention` | The model abstained (recognised no-answer phrasing) regardless of whether the question was answerable. |

The key consistency fix from the first draft is that the **Ignored-evidence**
label only applies when the gold context was actually the top-1 context the
generator read — not when it merely appears somewhere in the top-5. This
removes an asymmetric labelling in which rows where the generator never saw
the gold passage could be miscategorised as "ignored evidence".



In [19]:
def classify_error(row):
    abst = bool(row["rag_answer_abstained"])
    if row["is_answerable"]:
        if abst:
            return "No-answer / abstention"
        if row["rag_answer_em"] == 1 or row["rag_answer_f1"] >= CORRECT_F1_THRESHOLD:
            return "Correct"
        # The generator only sees top-1, so the success criterion must be
        # retrieval_success_top1, not retrieval_success_at_5.
        if not row["retrieval_success_top1"]:
            return "Retrieval miss / absent-evidence hallucination"
        return "Ignored-evidence hallucination"
    else:
        if abst:
            return "No-answer / abstention"
        return "Unanswerable failure"

results_df["error_type"] = results_df.apply(classify_error, axis=1)

error_counts = (results_df["error_type"]
                .value_counts()
                .rename_axis("error_type")
                .reset_index(name="count"))
error_counts["fraction"] = error_counts["count"] / len(results_df)
error_counts



,error_type,count,fraction
0,Correct,11,0.366667
1,Ignored-evidence hallucination,10,0.333333
2,Unanswerable failure,7,0.233333
3,Retrieval miss / absent-evidence hallucination,2,0.066667


### Hallucination categories — interpretation

- **Correct:** exact match or high token overlap with a gold answer.
- **Retrieval miss:** the gold context was not the top-1 passage and the model still produced an answer.
- **Ignored evidence:** the gold context was top-1, but the generated answer was wrong.
- **Unanswerable failure:** the question had no gold answer, but the model invented one.
- **No-answer / abstention:** the model declined to answer.

The labels are rule-based and useful for summarising dominant failure modes, but borderline paraphrases still require human inspection.

## Phase 2.7 — Representative Errors

Select up to two real examples from each error category, then fill remaining slots with the lowest-F1 failures. This keeps the table diverse without fabricating examples.

In [20]:
desired_types = [
    "Retrieval miss / absent-evidence hallucination",
    "Ignored-evidence hallucination",
    "Unanswerable failure",
    "No-answer / abstention",
    "Correct",
]

picked_indices = []
for error_type in desired_types:
    group = results_df[results_df["error_type"] == error_type]
    group = group.sort_values(["rag_answer_f1", "idx"], ascending=[True, True])
    picked_indices.extend(group.head(2).index.tolist())

# Keep unique rows and at most ten examples.
picked_indices = list(dict.fromkeys(picked_indices))[:10]
if len(picked_indices) < min(8, len(results_df)):
    remaining = results_df.drop(index=picked_indices)
    remaining = remaining[remaining["error_type"] != "Correct"].sort_values(
        ["rag_answer_f1", "idx"]
    )
    picked_indices.extend(remaining.head(10 - len(picked_indices)).index.tolist())

picked = results_df.loc[picked_indices].head(10)


def short(text, limit=160):
    text = (text or "").replace("\n", " ").strip()
    return text if len(text) <= limit else text[:limit] + "..."


EXPLANATIONS = {
    "Correct": "The answer exactly matches or closely overlaps a gold answer.",
    "Retrieval miss / absent-evidence hallucination":
        "The exact top-1 passage seen by the generator was not the gold context.",
    "Ignored-evidence hallucination":
        "The gold context was top-1, but the generated answer did not match the gold answer.",
    "Unanswerable failure":
        "SQuAD marks the question unanswerable, but the model produced a content answer.",
    "No-answer / abstention":
        "The model abstained; this is correct for unanswerable rows and a miss for answerable rows.",
}

rep_rows = []
for _, row in picked.iterrows():
    gold = row["gold_texts"][0] if row["gold_texts"] else "(unanswerable)"
    rep_rows.append({
        "Question": short(row["question"], 140),
        "Gold answer": short(gold, 80),
        "Retrieved context snippet (top-1)": short(row["rag_top_context"], 160),
        "RAG answer": short(row["rag_answer"], 80),
        "No-RAG answer": short(row["norag_answer"], 80),
        "Error type": row["error_type"],
        "Short explanation": EXPLANATIONS[row["error_type"]],
    })

rep_df = pd.DataFrame(rep_rows)
print("Representative examples:", len(rep_df))
rep_df

Representative examples: 8


,Question,Gold answer,Retrieved context snippet (top-1),RAG answer,No-RAG answer,Error type,Short explanation
0,What was the republican government amenable to?,war reparations,"In March 1971, the residential office of an FB...",assassinations of political activists,a reformed republic,Retrieval miss / absent-evidence hallucination,The exact top-1 passage seen by the generator ...
1,What are the helpers called that helped Buddha?,disciples,Buddhism /ˈbudɪzəm/ is a nontheistic religion[...,the awakened one,saigon,Retrieval miss / absent-evidence hallucination,The exact top-1 passage seen by the generator ...
2,Which recitation is the original basis of the ...,ʻAsim's,Vocalization markers indicating specific vowel...,Asim's recitation,sacrimony,Ignored-evidence hallucination,"The gold context was top-1, but the generated ..."
3,How many causalities did the US Air Force suff...,"68,000",The U.S. War Department created the first ante...,3,2,Ignored-evidence hallucination,"The gold context was top-1, but the generated ..."
4,What was the decrease in non-German population...,(unanswerable),"In July 2013, there were 41,000 non-Germans by...",24%,a year,Unanswerable failure,"SQuAD marks the question unanswerable, but the..."
5,In what year did Heinz Kloss develop a framewo...,(unanswerable),A framework was developed in 1967 by Heinz Klo...,1967,1897,Unanswerable failure,"SQuAD marks the question unanswerable, but the..."
6,Who established the Tibetan law code?,Tai Situ Changchub Gyaltsen,The Ming court appointed three Princes of Dhar...,Phagmodru ruler Tai Situ Changchub Gyaltsen,samuel sai,Correct,The answer exactly matches or closely overlaps...
7,How long did it take for Myanmar to recover fr...,250 years,Pagan's collapse was followed by 250 years of ...,250 years,a few days,Correct,The answer exactly matches or closely overlaps...


## Phase 2.8 — Limitations

- The evaluation subset is small, so the reported numbers are indicative rather than statistically rigorous.
- `flan-t5-small` is lightweight and weaker than modern instruction-tuned language models.
- The prompts are fixed and were not tuned.
- EM, Token F1, and ROUGE-L are lexical and may miss correct paraphrases.
- Hallucination labels are rule-based rather than human-annotated.
- RAG performance depends strongly on top-1 retrieval quality.
- Evaluation is conditioned on the gold context already being present in the small Phase 1 corpus.
- The model may fail to abstain on unanswerable questions.
- Phase 1 runs the generator on CPU; selecting a GPU alone does not move the model to CUDA.
- Library versions and hardware can slightly change generated outputs.

## Phase 2.9 — Future Work

The following improvements are future work and do not replace the Phase 1 baseline:

- strengthen the abstention prompt or separate answerability detection from answer generation;
- try a stronger embedding model such as `bge-base-en-v1.5` or `e5-base-v2`;
- add a lightweight cross-encoder reranker;
- pass multiple retrieved contexts to the generator with controlled truncation;
- evaluate more questions and add limited human review;
- test a stronger generator and a retrieval-confidence threshold;
- replace the SQuAD corpus with real course documents for YadYar Lite integration.

## Phase 2.10 — Lightweight Demo

The demo shows three distinct SQuAD examples. It displays the top retrieved contexts and similarity scores, while only rank 1 is passed to the unchanged Phase 1 RAG function.

In [21]:
import torch


def demo_row(row, k=3):
    question = row["question"]
    gold = row["answers"]["text"]
    print("=" * 78)
    print("Question:", question)
    print("Gold:", gold[0] if gold else "(unanswerable)")
    print("Answerable:", bool(gold))
    print(f"Top-{k} retrieved contexts (only rank 1 is passed to RAG):")

    for item in retrieve(question, k=k):
        cosine = 1.0 - item["score"] / 2.0
        snippet = item["context"].replace("\n", " ")[:160]
        print(f"  rank={item['rank']} cosine={cosine:.3f} | {snippet}...")

    with torch.no_grad():
        rag = generate_with_rag(question, k=RETRIEVE_K)
        norag = generate_without_rag(question)
    print("RAG answer:", rag["answer"])
    print("No-RAG answer:", norag["answer"])


answerable_demo_rows = [r for r in eval_rows if len(r["answers"]["text"]) > 0][:2]
unanswerable_demo_rows = [r for r in eval_rows if len(r["answers"]["text"]) == 0][:1]
for row in answerable_demo_rows + unanswerable_demo_rows:
    demo_row(row)

Question: How long did it take for Myanmar to recover from the collapse of it's first kingdom ?
Gold: 250 years
Answerable: True
Top-3 retrieved contexts (only rank 1 is passed to RAG):
  rank=1 cosine=0.562 | Pagan's collapse was followed by 250 years of political fragmentation that lasted well into the 16th century. Like the Burmans four centuries earlier, Shan migr...
  rank=2 cosine=0.367 | The history of biodiversity during the Phanerozoic (the last 540 million years), starts with rapid growth during the Cambrian explosion—a period during which ne...
  rank=3 cosine=0.316 | Tibet retained nominal power over religious and regional political affairs, while the Mongols managed a structural and administrative rule over the region, rein...
RAG answer: 250 years
No-RAG answer: a few days
Question: Which recitation is the original basis of the Quran of Cairo?
Gold: ʻAsim's
Answerable: True
Top-3 retrieved contexts (only rank 1 is passed to RAG):
  rank=1 cosine=0.495 | Vocalization marke

## Phase 2.11 — Input/Output and Future Integration Note

This component is designed to plug into the future YadYar Lite learning
assistant. No API server or deployment is built in Phase 2 — only the
contract that a future wrapper would honour.

### Sample input

```json
{
  "question": "What is backpropagation?",
  "course_documents": [
    "document text 1 ...",
    "document text 2 ..."
  ]
}
```

### Sample output

```json
{
  "answer": "Backpropagation is ...",
  "retrieved_contexts": [
    "...",
    "..."
  ],
  "retrieval_scores": [0.81, 0.74],
  "confidence_or_warning": "Evidence may be insufficient.",
  "error_type_if_known": "retrieval_miss"
}
```

### Notes

- `course_documents` would replace the current `contexts_list` derived from
  SQuAD; the embedding + FAISS index would be rebuilt over those documents.
- `retrieval_scores` are similarity scores derived from the FAISS L2
  distance (e.g. `1 - dist/2` on normalised vectors).
- `confidence_or_warning` would be set by a future abstention gate based on
  the retrieval score threshold described in Future Work.
- `error_type_if_known` is the rule-based category from Phase 2.6 — useful
  for downstream logging, not for blocking answers.
- Building an API or deployment is explicitly **out of scope** for this
  phase.


## Phase 2.12 — Save Outputs

Save the current run's metrics, predictions, breakdowns, and representative errors. List-valued columns are JSON-encoded before writing CSV files.

In [22]:
# ---- Save real Phase 2 outputs ----
out_cols = [
    "idx", "id", "question", "question_len_words", "is_answerable",
    "gold_texts", "gold_rank", "retrieval_success_top1",
    "retrieval_success_at_5", "rag_answer", "norag_answer",
    "rag_top_context", "rag_answer_em", "rag_answer_f1",
    "rag_answer_rougeL", "rag_answer_abstained", "norag_answer_em",
    "norag_answer_f1", "norag_answer_rougeL", "norag_answer_abstained",
    "error_type",
]

results_to_save = results_df[out_cols].copy()
results_to_save["gold_texts"] = results_to_save["gold_texts"].apply(json.dumps)
results_to_save.to_csv(os.path.join(OUTPUT_DIR, "phase2_results.csv"), index=False)
rep_df.to_csv(os.path.join(OUTPUT_DIR, "representative_errors.csv"), index=False)
breakdown_df.to_csv(os.path.join(OUTPUT_DIR, "breakdown_results.csv"), index=False)

metrics_summary = {
    "rag": aggregate(results_df, "rag_answer", "RAG"),
    "norag": aggregate(results_df, "norag_answer", "No-RAG"),
    "retrieval": retrieval_metrics,
    "n_eval": int(len(results_df)),
    "n_answerable": int(results_df["is_answerable"].sum()),
    "n_unanswerable": int((~results_df["is_answerable"]).sum()),
    "retrieve_k": RETRIEVE_K,
    "top_k_values": TOP_K_VALUES,
}

for filename, payload in [
    ("metrics_summary.json", metrics_summary),
    ("retrieval_metrics.json", retrieval_metrics),
]:
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

print("Saved:", sorted(os.listdir(OUTPUT_DIR)))


Saved: ['breakdown_results.csv', 'metrics_summary.json', 'phase2_results.csv', 'representative_errors.csv', 'retrieval_metrics.json']


## Phase 2.13 — Generate the Report and README

Generate `T05_Phase2_Report.md` and `README_T05_Phase2.md` from the current run after all metrics and representative examples are available.

In [23]:
# ---- Generate Markdown report and README from the current run ----
def fmt(value):
    return f"{value:.4f}" if isinstance(value, (int, float)) else str(value)

rag_m = metrics_summary["rag"]
norag_m = metrics_summary["norag"]
ret_m = retrieval_metrics

report_md = f"""# T-05 Phase 2 Report — RAG and Hallucination

## Project summary
Phase 2 evaluates the unchanged Phase 1 RAG and No-RAG baselines on a
closed-corpus SQuAD 2.0 subset.

## Baseline setup reminder
- Dataset: SQuAD 2.0 (`train`)
- Embeddings: `sentence-transformers/all-MiniLM-L6-v2`
- Search: FAISS flat L2 over {CORPUS_SIZE} contexts
- Generator: `google/flan-t5-small`
- RAG evidence: top-{RETRIEVE_K} context

## Dataset and evaluation subset
- Questions: {len(results_df)}
- Answerable: {int(results_df['is_answerable'].sum())}
- Unanswerable: {int((~results_df['is_answerable']).sum())}
- The gold context is inside the Phase 1 index by construction, so retrieval
  results measure ranking inside this selected corpus and are not estimates
  for the full SQuAD dataset.

## Retrieval results
| Metric | Value |
|---|---:|
| Recall@1 | {fmt(ret_m['Recall@1'])} |
| Recall@3 | {fmt(ret_m['Recall@3'])} |
| Recall@5 | {fmt(ret_m['Recall@5'])} |
| MRR | {fmt(ret_m['MRR'])} |

## Generation results
| Mode | EM | Token F1 | ROUGE-L | Answerable EM | Unanswerable detection |
|---|---:|---:|---:|---:|---:|
| RAG | {fmt(rag_m['EM'])} | {fmt(rag_m['Token F1'])} | {fmt(rag_m['ROUGE-L'])} | {fmt(rag_m['Answerable EM'])} | {fmt(rag_m['Unanswerable detection'])} |
| No-RAG | {fmt(norag_m['EM'])} | {fmt(norag_m['Token F1'])} | {fmt(norag_m['ROUGE-L'])} | {fmt(norag_m['Answerable EM'])} | {fmt(norag_m['Unanswerable detection'])} |

An all-zero No-RAG result is not changed or smoothed. The notebook displays
real No-RAG predictions so the result can be checked directly.

## Simple breakdown
{breakdown_df.to_markdown(index=False)}

## Representative errors
{rep_df.to_markdown(index=False)}

## Limitations
The subset and corpus are small; the generator and prompt are simple;
metrics are lexical; hallucination labels and abstention detection are
rule-based; and the closed-corpus sampling makes retrieval scores optimistic
relative to a full open-corpus evaluation.

## Future Work
Improve the abstention prompt, test a stronger embedder or reranker, increase
the corpus and evaluation set, try a stronger generator, and conduct limited
human review. These are future directions and do not replace the Phase 1
baseline in this phase.

## Lightweight demo summary
The notebook displays two distinct answerable questions and one unanswerable
question, their top retrieved contexts, RAG answer, No-RAG answer, and gold.

## Input/output integration note
Input: `{{"question": "...", "course_documents": ["..."]}}`

Output: `{{"answer": "...", "retrieved_contexts": ["..."],
"confidence_or_warning": "...", "error_type_if_known": "..."}}`

## External AI tools acknowledgement
External AI assistants were used for drafting, debugging, and writing support.
All code, outputs, and interpretations were reviewed by the team.
"""

readme_md = f"""# T-05 Phase 2 — Colab Instructions

1. Open `T05_Phase1_and_Phase2_RAG.ipynb` in Google Colab.
2. Run all cells from the beginning; Phase 1 must run before Phase 2.
3. Phase 1 installs `datasets`, `sentence-transformers`, `faiss-cpu`, and
   `transformers`.
4. `N_EVAL=30` is the fast setting; use 100 for a fuller CPU run.
5. Phase 1 keeps the model on CPU; selecting a GPU alone does not move it.
6. Outputs are written to `{OUTPUT_DIR}` and zipped after report generation.
7. Phase 2 does not replace or modify the Phase 1 baseline.
"""

for filename, text in [
    ("T05_Phase2_Report.md", report_md),
    ("README_T05_Phase2.md", readme_md),
]:
    with open(os.path.join(OUTPUT_DIR, filename), "w", encoding="utf-8") as f:
        f.write(text)
    print("Saved:", filename)


Saved: T05_Phase2_Report.md
Saved: README_T05_Phase2.md


## Phase 2.14 — ZIP Outputs

In [24]:
zip_base = "/content/T05_Phase2_Outputs"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Created:", zip_base + ".zip")
print("Included:", sorted(os.listdir(OUTPUT_DIR)))


Created: /content/T05_Phase2_Outputs.zip
Included: ['README_T05_Phase2.md', 'T05_Phase2_Report.md', 'breakdown_results.csv', 'metrics_summary.json', 'phase2_results.csv', 'representative_errors.csv', 'retrieval_metrics.json']


## Phase 2.15 — Short Presentation Outline (3–5 minutes)

1. Project question and Phase 1 baseline.
2. Retrieval results: Recall@1/3/5 and MRR.
3. RAG versus No-RAG generation metrics.
4. Main breakdowns and representative errors.
5. Unanswerable-question hallucinations.
6. Limitations and realistic Future Work.
7. Run the lightweight notebook demo.